# Redis Strings: The Foundation

WHAT IS A REDIS STRING?

  A Redis string is a binary-safe sequence of bytes (up to 512 MB).

  Despite the name "string", it can hold:
  
    - Text         ("hello world")
    - Numbers      (42, 3.14) — Redis auto-detects and allows INCR/DECR
    - Serialized   (JSON, MessagePack, Protocol Buffers)
    - Binary       (images, files — though not recommended for large ones)

  Strings are the simplest and most-used Redis type. Every other data
  structure is essentially built on top of strings.

PRODUCTION USE CASES:
  - Caching (API responses, DB query results, rendered HTML)
  - Counters (page views, API calls, rate limits)
  - Flags (feature toggles, maintenance mode)
  - Distributed locks (SET with NX and EX)
  - Session tokens

In [2]:
import sys, os, time, json
sys.path.insert(0, "..")

from connection_helper import get_redis, cleanup

In [3]:
PREFIX = "demo:strings:"

In [4]:
r = get_redis()

In [5]:
cleanup(r, PREFIX)

## 1. BASIC SET / GET
- SET stores a value; GET retrieves it.
- Time Complexity: O(1) - constant time regardless of value size.

In [6]:
r.set(f"{PREFIX}greeting", "Hello, Redis!")
value = r.get(f"{PREFIX}greeting")
print(f" SET + GET: {value}")

# GET on a non-existent key returns None (not an error)
missing = r.get(f"{PREFIX}does_not_exist")
print(f" Missing key returns: {missing} (type :{type(missing)})")

 SET + GET: Hello, Redis!
 Missing key returns: None (type :<class 'NoneType'>)


## 2. SET WITH OPTIONS (Critical for Prod)

SET has powerful options that makes it atomic:
- EX = Expire in N seconds
- PX = Expire in N milliseconds
- NX = set ONLY if key does NOT exist (used for locks!)
- XX = set ONLY if key ALREADY exists (used for updates)

In [ ]:
# EX: Auto-expire after 10 seconds (great for caching)
r.set(f"{PREFIX}cache_item", "cached_data", ex=10)
ttl = r.ttl(f"{PREFIX}cache_item")
print(f" Cached with TTL: {ttl} seconds remaining")

 Cached with TTL: 10 seconds remaining


In [12]:
# NX: Only set if doesn't already exist - this is how we will build locks!
# First attempt succeeds (key is new)
result1 = r.set(f"{PREFIX}lock", "owner_1", nx=True, ex=30)
print(f"Lock attempt 1(NX): {result1}") # True

# Second attempt fails (Key already exists)
result2 = r.set(f"{PREFIX}lock", "owner_2", nx=True, ex=30)
print(f"Lock attempt 2 (NX): {result2}") # None (failed!)

Lock attempt 1(NX): True
Lock attempt 2 (NX): None


In [14]:
# XX: Only set if it already exists - safe updates
r.set(f"{PREFIX}config", "v1")
r.set(f"{PREFIX}config", "v2", xx=True) 
print(f" XX Update existing: {r.get(f'{PREFIX}config')}")

r.set(f"{PREFIX}phantom", "value", xx=True) # FAILS (key doesn't exist)
print(f" XX on non-existent: {r.get(f'{PREFIX}phantom')}") # None

 XX Update existing: v2
 XX on non-existent: None


## 4. MULTI-KEY OPERATIONS (MSET / MGET)
- Set or Get multiple Kets in ONE round trip to Redis.
- Network round trip are often the bottleneck - MGET/MSET cut them down dramatically

- 10 individual GETS = 10 round trips ~ 10ms
- 1 MGET with 10 keys = 1 round trip ~ 1ms

In [25]:
r.mset({
    f"{PREFIX}user:1:name": "Alice",
    f"{PREFIX}user:1:email" : "alice@example.com",
    f"{PREFIX}user:1:role" : "admin",
})

True

In [26]:
values = r.mget({
    f"{PREFIX}user:1:name",
    f"{PREFIX}user:1:email",
    f"{PREFIX}user:1:role",
    f"{PREFIX}user:1:missing", # will be None
})

print(f"MGET results: {values}")

MGET results: ['admin', 'Alice', None, 'alice@example.com']


In [30]:
# MSETNX: set multiple keys ONLY if NONE of them exist
result = r.msetnx({
    f"{PREFIX}unique:a": "1",
    f"{PREFIX}unique:b": "2",
})

print(f" MSETNX (all new): {result}") # True

result = r.msetnx({
    f"{PREFIX}unique:a" : "changed", # Already exists!
    f"{PREFIX}unique:b" : "3",
})

print(f" MSETNX (one exists): {result}") # False - nothing was set

 MSETNX (all new): True
 MSETNX (one exists): False


## 5. STRING MANIPULATION

In [32]:
# APPEND: add to end of string
r.set(f"{PREFIX}log", "2024-01-01: started")
r.append(f"{PREFIX}log", " | 2024-01-02: updated")
print(f" APPEND: {r.get(f'{PREFIX}log')}")

 APPEND: 2024-01-01: started | 2024-01-02: updated


In [33]:
# STRLEN: get string length
print(f"STRLEN: {r.strlen(f'{PREFIX}log')} bytes")

STRLEN: 41 bytes


In [36]:
# GETRANGE: substring (0-indexed, inclusive on both ends)
print(f"GETRANGE [0:9]: {r.getrange(f'{PREFIX}log', 0, 9)}")

GETRANGE [0:9]: 2024-01-01


In [37]:
# SETRANGE: overwrite at offset
r.set(f"{PREFIX}padded", "Hello, World!")
r.setrange(f"{PREFIX}padded", 7, "Redis!")
print(f" SETRANGE: {r.get(f'{PREFIX}padded')}")

 SETRANGE: Hello, Redis!


In [41]:
# GETSET (now GETDEL in newer Redis): get old value, set new one
old = r.getset(f"{PREFIX}padded", "Completely new value")
print(f"GETSET old: '{old}', new: '{r.get(f'{PREFIX}padded')}'")

GETSET old: 'Completely new value', new: 'Completely new value'


## 6. STORING JSON (Common Pattern)
- Serialize on write, Deserialize on read

In [43]:
user = {
    "id": 42,
    "name": "Alice",
    "email": "alice@example.com",
    "preferences": {"theme": "dark", "lang": "en"},
}

r.set(f"{PREFIX}user:42:json", json.dumps(user), ex=3000)
cached = json.loads(r.get(f"{PREFIX}user:42:json"))
print(f"Stored & retrieved JSON: {cached['name']}, theme={cached['preferences']['theme']}")

Stored & retrieved JSON: Alice, theme=dark


## 7.SETNX PATTERN: Feature Flags

In [45]:
def is_feature_enabled(r, feature_name, default="false"):
    '''Check a feature flag stored in Redis'''
    key = f"{PREFIX}feature:{feature_name}"
    val = r.get(key)
    if val is None:
        r.set(key, default)
        return default == "true"
    return val == "true"

r.set(f"{PREFIX}feature:dark_mode", "true")
r.set(f"{PREFIX}feature:beta_ui", "false")

print(f" dark_mode enabled: {is_feature_enabled(r, 'dark_mode')}")
print(f" beta_ui enabled: {is_feature_enabled(r, 'beta_ui')}")
print(f" new_feature (auto): {is_feature_enabled(r, 'new_feature')}")

 dark_mode enabled: True
 beta_ui enabled: False
 new_feature (auto): False


## CLEANUP

In [46]:
cleanup(r, PREFIX)

 KEY TAKEAWAYS:                                         
                                                        
  • SET/GET is O(1) — the fastest operation in Redis     
  • Use EX for auto-expiring cache entries               
  • Use NX for locks and idempotent writes               
  • INCR/DECR are atomic — no race conditions            
  • MGET/MSET reduce network round trips                 
  • Store JSON for complex objects, Hashes for flat ones  